## CSAPP cachelab：把 s/E/b 换算成具体的数（1 KB 直接映射）

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.5（实践仓 `learning-408/csapp/cachelab/`）
Part B 评分 cache：`s = 5, E = 1, b = 5` →
- 组数 `S = 2^5 = 32`；相联度 `E = 1`（**直接映射**，冲突不命中的根源）
- 块大小 `B = 2^5 = 32` 字节 = **8 个 int**
- 总容量 `32 × 1 × 32 = 1024` 字节 = 1 KB（而 32×32 的 int 矩阵是 4 KB，装不下）

地址切分（写错一位全盘皆输）：
`tag = addr >> (s + b)`；`set_idx = (addr >> b) & ((1 << s) - 1)`

**模拟器要点**：`M` 是一次读 + 一次写（命中计 2 次 hit，未命中计 1 miss + 1 hit）；trace 里的 `I` 行要跳过；LRU 用时间戳，且**命中也要更新时间戳**。


## 分块（blocking）与冲突不命中：32×32 如何从 1183 降到 300 以下

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.5.3–7.5.5
**暴力转置为什么 1183 次 miss**：A 顺序访问 → 每 8 个 int 1 次 miss（约 128）；B 按列访问、步长 128 B → 组索引只落在 8 个组上；更要命的是 A 与 B 基址相差 4096 B = cache 容量的整数倍，**`A[i][j]` 与 `B[i][j]` 映射到同一组**，互相驱逐 → B 侧几乎每次都 miss。

**8×8 分块为什么有效**：① 工作集降到 16 个块，装得下；② 用 8 个局部变量一次读满 A 的一整行（8 int = 恰好一块），读入后立刻用完；③ B 的 8 次写落在 8 个不同组，不互相驱逐。残留问题只有**对角线块**（A/B 争同一组），单独处理即可。

**64×64 为什么 8×8 失效**（最有价值的一段推导）：一行 64 int = 256 B，组索引 = `(i*8 + j/8) mod 32`，**i 每加 4 组索引就绕回同一个值** → 8×8 块里 A 的第 5 行会把第 1 行挤掉，分块内部自己打架。解法是 8×8 外层 + 内部 4×4 的**两步搬运**：利用「`B[j+k][i..i+7]` 是同一个 32 B 块的两半」，把 A 的右上**先暂存到 B 的右上**（免费），第二阶段再搬到 B 的左下。

**408 对应**：Cache 三种映射、替换算法、**命中率手算**、局部性与分块 —— 组成原理大题重镇。


## [CS149] 多核 cache 一致性：MSI/MESI 与 false sharing

> 2026-10-22 · 来源：CS149 项目复习（真源 modules/408/CS149.md）

- **MSI 三态**：M（改过、全机唯一）/ S（与内存一致、可多副本、不能直接写）/ I（作废）。写 S 态必须先广播 invalidate —— **别人手里的副本越多，写越贵**。
- **MESI 的 E 态**：干净独占（只我读过、未改、与内存一致）。**E 升 M 是本地事务、不发总线** —— 私有的读-改-写不再骚扰总线，这就是加 E 的全部意义。
- **false sharing（伪共享）**：不同变量挤同一 64B cache line，各核写各自的变量，逻辑零冲突，但每次写都互相 invalidate，line 在核间乒乓，性能可掉一个数量级。修复：alignas(64) 或 padding 让每个热点变量独占一条 line。
- 与 408 的关系：408 只考到「多核 cache 一致性」概念级；协议级（MSI/MESI 状态转换）与工程级（false sharing 定位与修复）是 CS149 L14 的内容。
